In [66]:
import watermark
pkg_versions = watermark.watermark(
    packages="requests,pandas,tqdm,geopandas")
print(pkg_versions)

requests : 2.32.5
pandas   : 1.5.3
tqdm     : 4.67.1
geopandas: 1.0.1



In [65]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm
from io import StringIO
import xml.etree.ElementTree as ET
import geopandas as gpd

load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = "https://apis.data.go.kr"

# 행정안전부_행정표준코드_법정동코드

In [3]:
URL = f"{BASE_URL}/1741000/StanReginCd/getStanReginCdList"
params = {
    "serviceKey":API_KEY,
    "numOfRows": 1000,
    "pageNo": 1,
    "flag":"Y",
    "locatadd_nm":"부산광역시",
    "type":"json"
}
res = requests.get(URL, params= params)
data = res.json()
station_df = pd.DataFrame(data["StanReginCd"][1]["row"])
station_df["signguCode"] = (
    station_df["sido_cd"].astype(str).str.zfill(2)
    + station_df["sgg_cd"].astype(str).str.zfill(3)
)
busan_df = station_df[
    station_df["locallow_nm"].str.contains(
        r".*(?:구|군)$", na=False
    )
].reset_index(drop=True)

In [4]:
busan_df

,region_cd,sido_cd,sgg_cd,umd_cd,ri_cd,locatjumin_cd,locatjijuk_cd,locatadd_nm,locat_order,locat_rm,locathigh_cd,locallow_nm,adpt_de,signguCode
0,2611000000,26,110,000,00,2611000000,2611000000,부산광역시 중구,1,,2600000000,중구,,26110
1,2614000000,26,140,000,00,2614000000,2614000000,부산광역시 서구,2,,2600000000,서구,,26140
2,2617000000,26,170,000,00,2617000000,2617000000,부산광역시 동구,3,,2600000000,동구,,26170
3,2620000000,26,200,000,00,2620000000,2620000000,부산광역시 영도구,4,,2600000000,영도구,,26200
4,2623000000,26,230,000,00,2623000000,2623000000,부산광역시 부산진구,5,,2600000000,부산진구,,26230
5,2626000000,26,260,000,00,2626000000,2626000000,부산광역시 동래구,6,,2600000000,동래구,,26260
6,2629000000,26,290,000,00,2629000000,2629000000,부산광역시 남구,7,,2600000000,남구,,26290
7,2632000000,26,320,000,00,2632000000,2632000000,부산광역시 북구,8,,2600000000,북구,,26320
8,2635000000,26,350,000,00,2635000000,2635000000,부산광역시 해운대구,9,,2600000000,해운대구,,26350
9,2638000000,26,380,000,00,2638000000,2638000000,부산광역시 사하구,10,,2600000000,사하구,,26380


In [5]:
busan_codes = busan_df["signguCode"].unique()

# 부산광역시_부산버스정보시스템
- 정류소 정보
- 노선 정보

## 정류소정보

In [56]:
URL = f"{BASE_URL}/6260000/BusanBIMS/busStopList"

In [57]:
params = {
    "serviceKey":API_KEY,
    "numOfRows": 1,
    "pageNo": 1,
    # "_type":"json"
}

In [58]:
root = ET.fromstring(res.content)

total_count = int(root.findtext(".//totalCount"))
print(total_count)

8798


In [59]:
num_of_rows = 1000
params.update({"numOfRows":num_of_rows})

dfs = list()
for page in tqdm(range(1, total_count//num_of_rows + 2)):
    params.update({"pageNo":page})
    res = requests.get(URL, params= params)
    df = pd.read_xml(
        StringIO(res.text),
        xpath=".//item"
    )
    dfs.append(df)

100%|██████████| 9/9 [00:03<00:00,  2.31it/s]


In [60]:
df = pd.concat(dfs,ignore_index=True)
df.head()

,bstopid,bstopnm,arsno,gpsx,gpsy,stoptype
0,167970102,영주삼거리,1001.0,129.033322,35.115356,일반
1,169310303,영주삼거리,1002.0,129.033030,35.115283,일반
2,167970301,시민아파트,1003.0,129.031749,35.115140,일반
3,167840102,시민아파트,1004.0,129.032160,35.114921,일반
4,167970302,중앙공원.민주공원입구,1005.0,129.029762,35.114487,일반


In [70]:
bus_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["gpsx"], df["gpsy"]),
    crs="EPSG:4326"   # GPS 위경도
)

In [75]:
gdf = gpd.read_file("sigungu/sig.shp", encoding="cp949")
gdf = gdf.set_crs(epsg=5179)
gdf = gdf.to_crs(4326)

In [71]:
gdf

In [76]:
result = gpd.sjoin(
    bus_gdf,
    gdf,
    how="left",
    predicate="within"
)

In [88]:
result = gpd.sjoin_nearest(
    bus_gdf,
    gdf,
    how="left",
    distance_col="distance"
)

/home/conda/lib/python3.9/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


In [91]:
result.drop("geometry",axis=1)

,bstopid,bstopnm,arsno,gpsx,gpsy,stoptype,index_right,SIG_CD,SIG_ENG_NM,SIG_KOR_NM,distance
0,167970102,영주삼거리,1001.0,129.033322,35.115356,일반,25,26110,Jung-gu,중구,0.0
1,169310303,영주삼거리,1002.0,129.033030,35.115283,일반,25,26110,Jung-gu,중구,0.0
2,167970301,시민아파트,1003.0,129.031749,35.115140,일반,25,26110,Jung-gu,중구,0.0
3,167840102,시민아파트,1004.0,129.032160,35.114921,일반,25,26110,Jung-gu,중구,0.0
4,167970302,중앙공원.민주공원입구,1005.0,129.029762,35.114487,일반,25,26110,Jung-gu,중구,0.0
...,...,...,...,...,...,...,...,...,...,...,...
8801,519250000,김해건설공고,NaN,128.868122,35.276457,일반,216,48250,Gimhae-si,김해시,0.0
8802,519260000,유엔교차로,NaN,129.091960,35.131331,일반,31,26290,Nam-gu,남구,0.0
8803,519270000,유엔교차로,NaN,129.092221,35.131043,일반,31,26290,Nam-gu,남구,0.0
8804,519280000,팽나무공원,NaN,129.005902,35.084245,일반,34,26380,Saha-gu,사하구,0.0


In [92]:
merged_df = pd.merge(busan_df[["locallow_nm","signguCode"]], result.drop("geometry",axis=1), how="outer", left_on="signguCode", right_on="SIG_CD")
merged_df.shape

(8806, 13)

In [97]:
merged_df[~merged_df["signguCode"].isna()].reset_index(drop=True).to_csv("정류소정보.csv", index=False)

## 노선정보

In [99]:
URL = f"{BASE_URL}/6260000/BusanBIMS/busInfo"

In [102]:
params = {
    "serviceKey":API_KEY,
    # "numOfRows": 1,
    # "pageNo": 1,
    # "_type":"json"
}

In [103]:
root = ET.fromstring(res.content)

total_count = int(root.findtext(".//totalCount"))
print(total_count)

8798


In [104]:
num_of_rows = 1000
params.update({"numOfRows":num_of_rows})

dfs = list()
for page in tqdm(range(1, total_count//num_of_rows + 2)):
    params.update({"pageNo":page})
    res = requests.get(URL, params= params)
    df = pd.read_xml(
        StringIO(res.text),
        xpath=".//item"
    )
    dfs.append(df)

100%|██████████| 9/9 [00:02<00:00,  3.76it/s]


In [106]:
df = pd.concat(dfs,ignore_index=True)
df.head()

,lineid,buslinenum,bustype,startpoint,endpoint,companyid,headway,firsttime,endtime,headwaypeak,headwaynorm,headwayholi
0,5200010000,10,일반버스,연제공용차고지,감만현대아파트사거리,국제,7,04:30,22:15,12,12,16
1,5200100000,100,일반버스,청강리공영차고지,장전역,해동,8,04:50,21:50,15,16,19
2,5200100100,100-1,일반버스,송정,장전역,해동,8,05:00,22:10,17,19,21
3,5201001000,1001,급행버스,청강리공영차고지,하단,부일,9-10,04:30,22:00,11,12,12
4,5201002000,1002,급행버스,용당공영차고지,해운대구문화복합센터,삼신,20,04:50,22:00,17,17,19


In [107]:
df.to_csv("노선정보.csv",index=False)